In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing  import StandardScaler
from sklearn.cluster        import KMeans
from sklearn.metrics        import silhouette_score
import warnings, os
 
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
 
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
print("\n" + "="*70)
print("STEP 1 — LOADING DATA")
print("="*70)
 
features    = pd.read_csv(f"{OUTPUT_DIR}/model_ready_features.csv")
predictions = pd.read_csv(f"{OUTPUT_DIR}/churn_predictions.csv")
loyalty_v2  = pd.read_csv(f"{OUTPUT_DIR}/loyalty_cleaned_v2.csv")
 
# Merge everything onto one member-level table
members = features.merge(
    predictions[["loyalty_number", "churn_score", "risk_tier"]],
    on="loyalty_number", how="left"
).merge(
    loyalty_v2[["loyalty_number", "loyalty_card", "gender",
                "marital_status", "education", "province",
                "enrollment_year", "cancellation_year"]],
    on="loyalty_number", how="left"
)
 
print(f"  Members loaded: {len(members):,}")
print(f"  Columns: {members.shape[1]}")
 


STEP 1 — LOADING DATA
  Members loaded: 16,737
  Columns: 44


In [4]:
print("\n" + "="*70)
print("STEP 2 — STRATEGIC QUADRANT (CLV × Churn Risk)")
print("="*70)
 
# Use median CLV as the dividing line (more robust than mean given skew)
clv_median    = members["clv"].median()
churn_thresh  = 0.40   # medium+ risk threshold
 
print(f"  CLV median (dividing line)    : ${clv_median:,.2f}")
print(f"  Churn risk threshold          : {churn_thresh}")
 
def assign_quadrant(row):
    high_clv   = row["clv"]         >= clv_median
    high_risk  = row["churn_score"] >= churn_thresh
    if high_clv and not high_risk:
        return "Champions"          # High value, low risk → protect & reward
    elif high_clv and high_risk:
        return "At-Risk Stars"      # High value, high risk → urgent intervention
    elif not high_clv and not high_risk:
        return "Loyalists"          # Low value, low risk  → grow their value
    else:
        return "Sleeping/Lost"      # Low value, high risk → low-cost re-engage
 
members["strategic_quadrant"] = members.apply(assign_quadrant, axis=1)
 
quadrant_summary = members.groupby("strategic_quadrant").agg(
    count       = ("loyalty_number",  "count"),
    avg_clv     = ("clv",             "mean"),
    avg_score   = ("churn_score",     "mean"),
    avg_flights = ("flights_18m",     "mean"),
).round(2)
quadrant_summary["pct"] = (quadrant_summary["count"] /
                            len(members) * 100).round(1)
 
print(f"\n  Strategic Quadrant Distribution:")
print(quadrant_summary.to_string())
 


STEP 2 — STRATEGIC QUADRANT (CLV × Churn Risk)
  CLV median (dividing line)    : $5,780.18
  Churn risk threshold          : 0.4

  Strategic Quadrant Distribution:
                    count   avg_clv  avg_score  avg_flights   pct
strategic_quadrant                                               
At-Risk Stars        1859  11656.18       0.86         9.24  11.1
Champions            6511  12163.66       0.07        23.98  38.9
Loyalists            6528   3894.66       0.08        23.80  39.0
Sleeping/Lost        1839   4034.49       0.86         9.85  11.0


In [5]:
print("\n" + "="*70)
print("STEP 3 — BEHAVIOURAL CLUSTERING (KMeans)")
print("="*70)
 
# Features for clustering — behavioural signals only (no CLV, no churn score)
# We want clusters based on HOW they fly, not just their value
CLUSTER_FEATURES = [
    "flights_18m",          # overall frequency
    "active_months_18m",    # consistency
    "distance_18m",         # total distance (proxy for trip type)
    "redemption_ratio",     # how much they use their points
    "seasons_flown",        # seasonality
    "flight_trend",         # trajectory (growing vs declining)
    "months_since_last_flight",  # recency
    "pts_earned_18m",       # earning rate
]
 
# Use only members who have flown (exclude Dormant for clustering)
cluster_df = members[members["churn_label"] != 2][CLUSTER_FEATURES].copy()
cluster_ids = members[members["churn_label"] != 2]["loyalty_number"].values
 
print(f"  Clustering {len(cluster_df):,} active/churned members")
print(f"  Features: {CLUSTER_FEATURES}")
 
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_df.fillna(0))
 
# ── Find optimal K using silhouette score ─────────────────────────────────────
print(f"\n  Finding optimal K (silhouette score):")
silhouette_scores = {}
for k in range(3, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    score  = silhouette_score(X_scaled, labels, sample_size=5000, random_state=42)
    silhouette_scores[k] = score
    print(f"    K={k}: silhouette={score:.4f}")
 
best_k = max(silhouette_scores, key=silhouette_scores.get)
print(f"\n  Best K = {best_k} (silhouette = {silhouette_scores[best_k]:.4f})")
 
# Override to 5 if best_k is extreme — 5 gives good business interpretability
FINAL_K = best_k if 4 <= best_k <= 6 else 5
print(f"  Using K = {FINAL_K}")
 
# ── Final clustering ─────────────────────────────────────────────────────────
km_final = KMeans(n_clusters=FINAL_K, random_state=42, n_init=20)
cluster_labels = km_final.fit_predict(X_scaled)
 
# Add cluster labels back
cluster_map = dict(zip(cluster_ids, cluster_labels))
members["behaviour_cluster"] = members["loyalty_number"].map(cluster_map)
members["behaviour_cluster"] = members["behaviour_cluster"].fillna(-1).astype(int)
# -1 = Dormant (not clustered)
 
# ── Cluster profiles ──────────────────────────────────────────────────────────
cluster_profile = (members[members["behaviour_cluster"] >= 0]
                   .groupby("behaviour_cluster")
                   .agg(
                       count                    = ("loyalty_number",           "count"),
                       avg_clv                  = ("clv",                      "mean"),
                       avg_churn_score          = ("churn_score",              "mean"),
                       avg_flights_18m          = ("flights_18m",              "mean"),
                       avg_active_months        = ("active_months_18m",        "mean"),
                       avg_distance_18m         = ("distance_18m",             "mean"),
                       avg_redemption_ratio     = ("redemption_ratio",         "mean"),
                       avg_seasons_flown        = ("seasons_flown",            "mean"),
                       avg_flight_trend         = ("flight_trend",             "mean"),
                       avg_months_since_flight  = ("months_since_last_flight", "mean"),
                       avg_pts_earned           = ("pts_earned_18m",           "mean"),
                   ).round(2))
 
print(f"\n  Cluster Profiles:")
print(cluster_profile.to_string())
 
# ── Auto-name clusters based on profile characteristics ──────────────────────
def name_cluster(row):
    flights  = row["avg_flights_18m"]
    trend    = row["avg_flight_trend"]
    recency  = row["avg_months_since_flight"]
    redeem   = row["avg_redemption_ratio"]
    seasons  = row["avg_seasons_flown"]
    clv      = row["avg_clv"]
 
    if recency >= 8:
        return "Fading Flyers"        # Haven't flown in a long time
    elif flights >= 15 and trend >= 0:
        return "Road Warriors"        # Very frequent, stable or growing
    elif flights >= 8 and seasons >= 3:
        return "Frequent Travellers"  # Regular, multi-season flyers
    elif redeem >= 0.15 and flights >= 3:
        return "Points Collectors"    # Moderate flyers, engaged with rewards
    elif trend < -1 and flights < 5:
        return "Declining Casuals"    # Infrequent and flying less
    else:
        return "Occasional Flyers"    # Low frequency, sporadic
 
cluster_profile["segment_name"] = cluster_profile.apply(name_cluster, axis=1)
print(f"\n  Cluster Names:")
for idx, row in cluster_profile.iterrows():
    print(f"    Cluster {idx}: {row['segment_name']:25s} "
          f"(n={row['count']:,}, avg_flights={row['avg_flights_18m']:.1f}, "
          f"churn={row['avg_churn_score']:.2f})")
 
# Map names back to members
cluster_name_map = cluster_profile["segment_name"].to_dict()
cluster_name_map[-1] = "Dormant"
members["segment_name"] = members["behaviour_cluster"].map(cluster_name_map)
 


STEP 3 — BEHAVIOURAL CLUSTERING (KMeans)
  Clustering 15,167 active/churned members
  Features: ['flights_18m', 'active_months_18m', 'distance_18m', 'redemption_ratio', 'seasons_flown', 'flight_trend', 'months_since_last_flight', 'pts_earned_18m']

  Finding optimal K (silhouette score):
    K=3: silhouette=0.2724
    K=4: silhouette=0.2959
    K=5: silhouette=0.3279
    K=6: silhouette=0.2806
    K=7: silhouette=0.2834
    K=8: silhouette=0.2832

  Best K = 5 (silhouette = 0.3279)
  Using K = 5

  Cluster Profiles:
                   count  avg_clv  avg_churn_score  avg_flights_18m  avg_active_months  avg_distance_18m  avg_redemption_ratio  avg_seasons_flown  avg_flight_trend  avg_months_since_flight  avg_pts_earned
behaviour_cluster                                                                                                                                                                                           
0                    543  8168.43             0.08            25.18

In [6]:
print("\n" + "="*70)
print("STEP 4 — COMBINED SEGMENT LABELS")
print("="*70)
 
# Final segment = strategic quadrant for all members
# (behaviour cluster adds detail within each quadrant)
members["final_segment"] = members["strategic_quadrant"].copy()
members.loc[members["churn_label"] == 2, "final_segment"] = "Dormant"
 
segment_summary = members.groupby("final_segment").agg(
    count       = ("loyalty_number",           "count"),
    avg_clv     = ("clv",                      "mean"),
    med_clv     = ("clv",                      "median"),
    avg_score   = ("churn_score",              "mean"),
    avg_flights = ("flights_18m",              "mean"),
    avg_tenure  = ("enrollment_tenure_months", "mean"),
).round(2)
segment_summary["pct"] = (segment_summary["count"] /
                           len(members) * 100).round(1)
 
print(f"\n  Final Segment Summary:")
print(segment_summary.to_string())
 


STEP 4 — COMBINED SEGMENT LABELS

  Final Segment Summary:
               count   avg_clv  med_clv  avg_score  avg_flights  avg_tenure   pct
final_segment                                                                    
At-Risk Stars   1057  10951.94  8581.28       0.75        16.25       25.65   6.3
Champions       6511  12163.66  9032.37       0.07        23.98       32.98  38.9
Dormant         1570   8360.82  5907.95       1.00         0.00       36.68   9.4
Loyalists       6528   3894.66  3920.77       0.08        23.80       33.02  39.0
Sleeping/Lost   1071   4094.83  4285.58       0.76        16.92       28.24   6.4


In [7]:
print("\n" + "="*70)
print("STEP 5 — RETENTION ACTION MATRIX")
print("="*70)
 
retention_actions = {
    "Champions": {
        "priority"   : "HIGH — Protect",
        "trigger"    : "Proactive — quarterly touchpoint",
        "action"     : "Exclusive tier upgrade preview + personalised route offer",
        "channel"    : "Email + App notification",
        "timing"     : "Before their typical booking season",
        "kpi"        : "Retain 95%+ annually"
    },
    "At-Risk Stars": {
        "priority"   : "URGENT — Intervene now",
        "trigger"    : "Churn score >= 0.40 AND CLV >= median",
        "action"     : "Personal outreach from loyalty team + status match offer "
                       "+ 2x points on next 3 bookings",
        "channel"    : "Phone call + personalised email within 48 hrs",
        "timing"     : "Immediately when score crosses 0.40",
        "kpi"        : "Recover 30%+ of flagged members within 90 days"
    },
    "Loyalists": {
        "priority"   : "MEDIUM — Grow",
        "trigger"    : "Low churn risk but CLV below median",
        "action"     : "Tier upgrade challenge: earn X points in 60 days to "
                       "reach next tier + bonus points on underutilised routes",
        "channel"    : "Email + in-app challenge notification",
        "timing"     : "At 6-month enrollment anniversary",
        "kpi"        : "Increase avg CLV by 15% within 12 months"
    },
    "Sleeping/Lost": {
        "priority"   : "LOW — Efficient re-engage",
        "trigger"    : "Churn score >= 0.40 AND CLV < median",
        "action"     : "'We miss you' campaign — discounted fare on their "
                       "most-flown route, expiry in 30 days",
        "channel"    : "Email only (low cost)",
        "timing"     : "Month 4 of inactivity",
        "kpi"        : "5% reactivation rate within 60 days"
    },
    "Dormant": {
        "priority"   : "LOW — Activation",
        "trigger"    : "Zero flights in 2017–2018",
        "action"     : "First-flight incentive: double points + waived booking fee "
                       "on first booking within 90 days",
        "channel"    : "Email sequence (3 touches over 30 days)",
        "timing"     : "Month 3 post-enrollment if no flight booked",
        "kpi"        : "10% first-flight conversion"
    }
}
 
print(f"\n  RETENTION PLAYBOOK:")
print(f"  {'='*65}")
for segment, actions in retention_actions.items():
    count = (members["final_segment"] == segment).sum()
    print(f"\n  [{segment}]  ({count:,} members)")
    for k, v in actions.items():
        print(f"    {k:<12}: {v}")
 


STEP 5 — RETENTION ACTION MATRIX

  RETENTION PLAYBOOK:

  [Champions]  (6,511 members)
    priority    : HIGH — Protect
    trigger     : Proactive — quarterly touchpoint
    action      : Exclusive tier upgrade preview + personalised route offer
    channel     : Email + App notification
    timing      : Before their typical booking season
    kpi         : Retain 95%+ annually

  [At-Risk Stars]  (1,057 members)
    priority    : URGENT — Intervene now
    trigger     : Churn score >= 0.40 AND CLV >= median
    action      : Personal outreach from loyalty team + status match offer + 2x points on next 3 bookings
    channel     : Phone call + personalised email within 48 hrs
    timing      : Immediately when score crosses 0.40
    kpi         : Recover 30%+ of flagged members within 90 days

  [Loyalists]  (6,528 members)
    priority    : MEDIUM — Grow
    trigger     : Low churn risk but CLV below median
    action      : Tier upgrade challenge: earn X points in 60 days to reach

In [8]:
print("\n" + "="*70)
print("STEP 6 — SAVING SEGMENT PLOTS")
print("="*70)
 
QUAD_COLORS = {
    "Champions"    : "#2196F3",   # blue
    "At-Risk Stars": "#F44336",   # red
    "Loyalists"    : "#4CAF50",   # green
    "Sleeping/Lost": "#FF9800",   # orange
    "Dormant"      : "#9E9E9E",   # grey
}
 
# Plot 1: CLV × Churn Score scatter (strategic quadrant map)
fig, ax = plt.subplots(figsize=(11, 7))
for seg, color in QUAD_COLORS.items():
    subset = members[members["final_segment"] == seg]
    ax.scatter(subset["churn_score"], subset["clv"].clip(upper=30000),
               c=color, label=seg, alpha=0.35, s=12, edgecolors="none")
 
ax.axvline(churn_thresh, color="black", linestyle="--", linewidth=1, alpha=0.6)
ax.axhline(clv_median,   color="black", linestyle="--", linewidth=1, alpha=0.6)
ax.set_xlabel("Churn Score (probability of churning)", fontsize=11)
ax.set_ylabel("CLV (capped at $30,000)", fontsize=11)
ax.set_title("Customer Strategic Quadrant Map\nCLV × Churn Risk",
             fontweight="bold", fontsize=13)
 
# Quadrant labels
ax.text(0.05, 28000, "Champions",     fontsize=10, color="#2196F3", fontweight="bold")
ax.text(0.55, 28000, "At-Risk Stars", fontsize=10, color="#F44336", fontweight="bold")
ax.text(0.05, 500,   "Loyalists",     fontsize=10, color="#4CAF50", fontweight="bold")
ax.text(0.55, 500,   "Sleeping/Lost", fontsize=10, color="#FF9800", fontweight="bold")
 
legend_patches = [mpatches.Patch(color=c, label=s)
                  for s, c in QUAD_COLORS.items()]
ax.legend(handles=legend_patches, loc="upper center",
          bbox_to_anchor=(0.5, -0.10), ncol=5, frameon=False)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plot_segment_quadrant_map.png", dpi=130,
            bbox_inches="tight")
plt.close()
print(f"  Saved: plot_segment_quadrant_map.png")
 
# Plot 2: Segment size and avg CLV
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
seg_order = ["Champions", "At-Risk Stars", "Loyalists", "Sleeping/Lost", "Dormant"]
colors    = [QUAD_COLORS[s] for s in seg_order]
 
counts = [segment_summary.loc[s, "count"] if s in segment_summary.index else 0
          for s in seg_order]
avg_clvs = [segment_summary.loc[s, "avg_clv"] if s in segment_summary.index else 0
            for s in seg_order]
 
axes[0].bar(seg_order, counts, color=colors, edgecolor="white")
axes[0].set_title("Members per Segment", fontweight="bold")
axes[0].set_ylabel("Members")
axes[0].tick_params(axis="x", rotation=15)
for i, v in enumerate(counts):
    axes[0].text(i, v + 30, f"{v:,}", ha="center", fontsize=9)
 
axes[1].bar(seg_order, avg_clvs, color=colors, edgecolor="white")
axes[1].set_title("Average CLV per Segment", fontweight="bold")
axes[1].set_ylabel("Average CLV ($)")
axes[1].tick_params(axis="x", rotation=15)
for i, v in enumerate(avg_clvs):
    axes[1].text(i, v + 50, f"${v:,.0f}", ha="center", fontsize=9)
 
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plot_segment_size_clv.png", dpi=120)
plt.close()
print(f"  Saved: plot_segment_size_clv.png")
 
# Plot 3: Behavioural cluster radar chart
# Normalise cluster profile for radar
radar_features = ["avg_flights_18m", "avg_active_months", "avg_redemption_ratio",
                  "avg_seasons_flown", "avg_pts_earned"]
radar_labels   = ["Flights\n(18m)", "Active\nMonths", "Redemption\nRatio",
                  "Seasons\nFlown", "Points\nEarned"]
 
profile_norm = cluster_profile[radar_features].copy()
for col in radar_features:
    col_min = profile_norm[col].min()
    col_max = profile_norm[col].max()
    if col_max > col_min:
        profile_norm[col] = (profile_norm[col] - col_min) / (col_max - col_min)
    else:
        profile_norm[col] = 0.5
 
N = len(radar_features)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]
 
cluster_colors = plt.cm.tab10(np.linspace(0, 0.7, FINAL_K))
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
 
for idx, (cluster_id, row) in enumerate(profile_norm.iterrows()):
    values = row[radar_features].tolist()
    values += values[:1]
    name   = cluster_name_map.get(cluster_id, f"Cluster {cluster_id}")
    ax.plot(angles, values, color=cluster_colors[idx], linewidth=2, label=name)
    ax.fill(angles, values, color=cluster_colors[idx], alpha=0.1)
 
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=10)
ax.set_title("Behavioural Cluster Profiles\n(normalised)", fontweight="bold",
             pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plot_segment_cluster_radar.png", dpi=120,
            bbox_inches="tight")
plt.close()
print(f"  Saved: plot_segment_cluster_radar.png")
 
# Plot 4: Churn score distribution per segment
fig, ax = plt.subplots(figsize=(11, 5))
for seg in ["Champions", "At-Risk Stars", "Loyalists", "Sleeping/Lost"]:
    subset = members[members["final_segment"] == seg]["churn_score"]
    ax.hist(subset, bins=30, alpha=0.5, label=seg,
            color=QUAD_COLORS[seg], density=True, edgecolor="none")
ax.axvline(churn_thresh, color="black", linestyle="--",
           linewidth=1.2, label=f"Risk threshold ({churn_thresh})")
ax.set_title("Churn Score Distribution by Segment", fontweight="bold")
ax.set_xlabel("Churn Score")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plot_segment_score_distribution.png", dpi=120)
plt.close()
print(f"  Saved: plot_segment_score_distribution.png")
 


STEP 6 — SAVING SEGMENT PLOTS
  Saved: plot_segment_quadrant_map.png
  Saved: plot_segment_size_clv.png
  Saved: plot_segment_cluster_radar.png
  Saved: plot_segment_score_distribution.png


In [9]:
print("\n" + "="*70)
print("STEP 7 — SAVING OUTPUTS")
print("="*70)
 
# Full member-level segmented file
save_cols = [
    "loyalty_number", "loyalty_card", "clv", "province",
    "enrollment_year", "gender", "marital_status", "education",
    "churn_score", "risk_tier", "final_segment", "segment_name",
    "strategic_quadrant", "behaviour_cluster",
    "flights_18m", "active_months_18m", "months_since_last_flight",
    "flight_trend", "redemption_ratio", "seasons_flown",
    "pts_earned_18m", "distance_18m", "enrollment_tenure_months",
    "churn_label"
]
save_cols = [c for c in save_cols if c in members.columns]
members[save_cols].to_csv(f"{OUTPUT_DIR}/segmented_members.csv", index=False)
print(f"  Saved: segmented_members.csv  ({len(members):,} members)")
 
# Segment profile summary
segment_profile_full = members.groupby("final_segment").agg(
    count                   = ("loyalty_number",           "count"),
    avg_clv                 = ("clv",                      "mean"),
    median_clv              = ("clv",                      "median"),
    avg_churn_score         = ("churn_score",              "mean"),
    avg_flights_18m         = ("flights_18m",              "mean"),
    avg_months_since_flight = ("months_since_last_flight", "mean"),
    avg_redemption_ratio    = ("redemption_ratio",         "mean"),
    avg_tenure_months       = ("enrollment_tenure_months", "mean"),
    pct_aurora_card         = ("loyalty_card",
                               lambda x: (x == "Aurora").mean() * 100),
).round(2)
 
segment_profile_full.to_csv(f"{OUTPUT_DIR}/segment_profiles.csv")
print(f"  Saved: segment_profiles.csv")
print(f"\n  Full Segment Profiles:")
print(segment_profile_full.to_string())
 
print(f"""
{"="*70}
PHASE 4 COMPLETE
{"="*70}
 
  SEGMENTS CREATED:
  ─────────────────────────────────────────────────────
  Champions     — High CLV, low churn risk   → Protect & reward
  At-Risk Stars — High CLV, high churn risk  → Urgent intervention
  Loyalists     — Low CLV, low churn risk    → Grow their value
  Sleeping/Lost — Low CLV, high churn risk   → Low-cost re-engage
  Dormant       — Never flew                 → Activation campaign
  ─────────────────────────────────────────────────────
 
  NEXT STEP → Phase 5: Streamlit Dashboard
  Run: streamlit run phase5_dashboard.py
""")
 


STEP 7 — SAVING OUTPUTS
  Saved: segmented_members.csv  (16,737 members)
  Saved: segment_profiles.csv

  Full Segment Profiles:
               count   avg_clv  median_clv  avg_churn_score  avg_flights_18m  avg_months_since_flight  avg_redemption_ratio  avg_tenure_months  pct_aurora_card
final_segment                                                                                                                                                  
At-Risk Stars   1057  10951.94     8581.28             0.75            16.25                     4.34                  0.02              25.65            27.53
Champions       6511  12163.66     9032.37             0.07            23.98                     0.43                  0.02              32.98            33.19
Dormant         1570   8360.82     5907.95             1.00             0.00                     0.00                  0.00              36.68            20.83
Loyalists       6528   3894.66     3920.77             0.08           